<a href="https://colab.research.google.com/github/beingmen4-web/python-project/blob/main/Hospital_Analysis_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hospital Appointment and Patient Flow Analysis

**Before running:** upload `hospital_data.csv` in Colab (folder icon on the left, then upload icon).
Then click `Runtime > Run all`. The code reads the data from `/content/hospital_data.csv`.

In [12]:
import numpy as np
import pandas as pd


In [13]:
# Step 1: read the data from the uploaded csv file
import os

# Check if the file has a suffix due to duplicate uploads
if os.path.exists("/content/hospital_data (2).csv"):
    file_path = "/content/hospital_data (2).csv"
else:
    file_path = "/content/hospital_data.csv"

df = pd.read_csv(file_path)
print(f"Successfully loaded: {file_path}")
print(df.head())

Successfully loaded: /content/hospital_data (2).csv
   patient_id  age  gender  department      doctor        day  hour  \
0           1   38  Female     General  Dr. Sharma   Thursday    12   
1           2   13    Male        Skin   Dr. Gupta  Wednesday    13   
2           3   73  Female     General    Dr. Khan  Wednesday    16   
3           4   10    Male        Skin   Dr. Singh  Wednesday    12   
4           5   76    Male  Pediatrics    Dr. Khan     Monday    12   

   booking_gap_days  sms_sent     status  waiting_time  consult_time  \
0                12         1  Completed          34.0          22.0   
1                 6         0    No-Show           NaN           NaN   
2                19         1  Completed          14.0           8.0   
3                 0         0  Completed          24.0          15.0   
4                22         1  Completed          33.0          11.0   

  age_group  no_show  total_time   gap_type  
0     Adult        0        56.0  Short ga

In [14]:
# Step 2: check the data
print("\nShape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\n", df.describe())



Shape: (100, 16)

Missing values:
patient_id           0
age                  0
gender               0
department           0
doctor               0
day                  0
hour                 0
booking_gap_days     0
sms_sent             0
status               0
waiting_time        18
consult_time        18
age_group            0
no_show              0
total_time          18
gap_type             0
dtype: int64

        patient_id         age        hour  booking_gap_days    sms_sent  \
count  100.000000  100.000000  100.000000        100.000000  100.000000   
mean    50.500000   40.290000   12.770000         14.920000    0.540000   
std     29.011492   26.891212    2.168903          8.396344    0.500908   
min      1.000000    1.000000    9.000000          0.000000    0.000000   
25%     25.750000   15.750000   11.000000          7.750000    0.000000   
50%     50.500000   34.000000   13.000000         16.000000    1.000000   
75%     75.250000   68.250000   15.000000         22.0000

In [15]:
# Step 3: new columns
df["age_group"] = pd.cut(df["age"], bins=[0, 14, 30, 50, 100],
                         labels=["Child", "Young", "Adult", "Senior"])
df["no_show"] = np.where(df["status"] == "No-Show", 1, 0)
df["total_time"] = df["waiting_time"] + df["consult_time"]
df["gap_type"] = np.where(df["booking_gap_days"] > 15, "Long gap", "Short gap")

# only the patients who actually came
completed = df[df["status"] == "Completed"]


In [16]:
# Step 4: main numbers
total = len(df)
noshow_count = df["no_show"].sum()

print("\n--- Main Numbers ---")
print("Total appointments:", total)
print("Completed:", total - noshow_count)
print("No-shows:", noshow_count)
print("No-show percentage:", round(noshow_count / total * 100, 2), "%")
print("Average waiting time:", round(completed["waiting_time"].mean(), 2), "min")
print("Maximum waiting time:", completed["waiting_time"].max(), "min")
print("Average consultation time:", round(completed["consult_time"].mean(), 2), "min")
print("Average total time in hospital:", round(completed["total_time"].mean(), 2), "min")

long_wait = completed[completed["waiting_time"] > 30]
print("Patients who waited more than 30 min:", len(long_wait), "out of", len(completed))



--- Main Numbers ---
Total appointments: 100
Completed: 82
No-shows: 18
No-show percentage: 18.0 %
Average waiting time: 20.6 min
Maximum waiting time: 39.0 min
Average consultation time: 15.3 min
Average total time in hospital: 35.9 min
Patients who waited more than 30 min: 13 out of 82


In [17]:
# Step 5: busiest day and hour
print("\n--- Appointments per day ---")
print(df["day"].value_counts())
print("\n--- Appointments per hour ---")
print(df["hour"].value_counts().sort_index())
print("\n--- Day vs Hour table ---")
print(pd.crosstab(df["day"], df["hour"]))



--- Appointments per day ---
day
Wednesday    22
Monday       21
Friday       17
Thursday     14
Saturday     14
Tuesday      12
Name: count, dtype: int64

--- Appointments per hour ---
hour
9     10
10     7
11    12
12    18
13    12
14    14
15    15
16    12
Name: count, dtype: int64

--- Day vs Hour table ---
hour       9   10  11  12  13  14  15  16
day                                      
Friday      1   0   2   2   3   2   4   3
Monday      1   2   4   6   3   2   3   0
Saturday    2   0   2   2   2   4   1   1
Thursday    1   0   4   1   1   1   3   3
Tuesday     2   2   0   3   0   3   2   0
Wednesday   3   3   0   4   3   2   2   5


In [18]:
# Step 6: no-show analysis
print("\n--- No-show % by sms (0 = no sms, 1 = sms sent) ---")
print(df.groupby("sms_sent")["no_show"].mean() * 100)
print("\n--- No-show % by department ---")
print((df.groupby("department")["no_show"].mean() * 100).round(2))
print("\n--- No-show % by age group ---")
print((df.groupby("age_group")["no_show"].mean() * 100).round(2))
print("\n--- No-show % by booking gap ---")
print((df.groupby("gap_type")["no_show"].mean() * 100).round(2))



--- No-show % by sms (0 = no sms, 1 = sms sent) ---
sms_sent
0    23.913043
1    12.962963
Name: no_show, dtype: float64

--- No-show % by department ---
department
Cardiology      7.69
ENT            23.53
General         9.52
Orthopedics    26.67
Pediatrics     21.05
Skin           20.00
Name: no_show, dtype: float64

--- No-show % by age group ---
age_group
Child     29.17
Young     17.39
Adult      0.00
Senior    17.07
Name: no_show, dtype: float64

--- No-show % by booking gap ---
gap_type
Long gap     23.08
Short gap    12.50
Name: no_show, dtype: float64


/tmp/ipykernel_439/277809563.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print((df.groupby("age_group")["no_show"].mean() * 100).round(2))


In [19]:
# Step 7: waiting time analysis
print("\n--- Average waiting time by hour ---")
print(completed.groupby("hour")["waiting_time"].mean().round(2))
print("\n--- Average waiting time by department ---")
print(completed.groupby("department")["waiting_time"].mean().round(2))
print("\n--- Average waiting time by day ---")
print(completed.groupby("day")["waiting_time"].mean().round(2))



--- Average waiting time by hour ---
hour
9     13.67
10    26.80
11    28.30
12    31.19
13    16.43
14    14.18
15    15.46
16    16.18
Name: waiting_time, dtype: float64

--- Average waiting time by department ---
department
Cardiology     15.00
ENT            21.92
General        22.42
Orthopedics    23.64
Pediatrics     20.00
Skin           19.83
Name: waiting_time, dtype: float64

--- Average waiting time by day ---
day
Friday       17.79
Monday       22.35
Saturday     19.17
Thursday     21.82
Tuesday      24.00
Wednesday    19.47
Name: waiting_time, dtype: float64


In [20]:
# Step 8: doctor workload
doctor_table = completed.groupby("doctor").agg(
    patients_seen=("patient_id", "count"),
    avg_consult=("consult_time", "mean"),
    avg_wait=("waiting_time", "mean")
)
print("\n--- Doctor workload ---")
print(doctor_table.round(2))



--- Doctor workload ---
            patients_seen  avg_consult  avg_wait
doctor                                          
Dr. Gupta              18        16.11     19.06
Dr. Khan               17        14.24     22.82
Dr. Sharma             18        16.22     19.94
Dr. Singh              18        14.06     20.78
Dr. Verma              11        16.18     20.45


In [21]:
# Step 9: conclusion
busy_day = df["day"].value_counts().idxmax()
busy_hour = df["hour"].value_counts().idxmax()
worst_wait_hour = completed.groupby("hour")["waiting_time"].mean().idxmax()
sms = df.groupby("sms_sent")["no_show"].mean() * 100

print("\n--- Conclusion ---")
print("Busiest day:", busy_day)
print("Busiest hour:", busy_hour)
print("Hour with longest waiting:", worst_wait_hour)
print("No-show % without sms:", round(sms[0], 2))
print("No-show % with sms:", round(sms[1], 2))
print("\nWhat the hospital can do:")
print("1. Send sms reminder to every patient, it reduces no-shows.")
print("2. Keep more staff between 10 and 12 when waiting time is high.")
print("3. Patients who book too early miss more, so call them once before the visit.")



--- Conclusion ---
Busiest day: Wednesday
Busiest hour: 12
Hour with longest waiting: 12
No-show % without sms: 23.91
No-show % with sms: 12.96

What the hospital can do:
1. Send sms reminder to every patient, it reduces no-shows.
2. Keep more staff between 10 and 12 when waiting time is high.
3. Patients who book too early miss more, so call them once before the visit.


In [22]:
# Step 10: save the doctor table
doctor_table.to_csv("/content/doctor_table.csv")
print("\nFile saved: /content/doctor_table.csv")


File saved: /content/doctor_table.csv
